# Face Occlusion — Kaggle Training Notebook

## 1. Install & Clone

In [ ]:
import os

# Install PyTorch compatible with P100 (sm_60, CUDA 11.8)
!pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q --upgrade timm wandb tqdm

!git clone https://github.com/LeHoang510/Face-Occlusion-Prediction.git
os.chdir("/kaggle/working/Face-Occlusion-Prediction")

import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Working dir:", os.getcwd())

## 2. Prepare Dataset

In [ ]:
import os

KAGGLE_INPUT = "/kaggle/input/datasets/lhongnguyn/face-occlusion-data"

img_root  = f"{KAGGLE_INPUT}/crops/Crop_224_5fp_100K"
train_csv = f"{KAGGLE_INPUT}/DataChallenge2026/occlusion_datasets/train.csv"
test_csv  = f"{KAGGLE_INPUT}/DataChallenge2026/occlusion_datasets/test_students.csv"

assert os.path.isdir(img_root),   f"Not found: {img_root}"
assert os.path.isfile(train_csv), f"Not found: {train_csv}"
assert os.path.isfile(test_csv),  f"Not found: {test_csv}"

print("All paths OK.")
print("img_root :", img_root)
print("train_csv:", train_csv)
print("test_csv :", test_csv)

## 3. WandB Login

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("WandB API key loaded.")

## 4. Configure & Train

Edit `CONFIG` below to pick the model you want to train.

In [ ]:
import yaml

# ── Pick your config ──────────────────────────────────────────────────
CONFIG = "src/data_challenge/configs/dinov2_small.yaml"
# CONFIG = "src/data_challenge/configs/dinov2_base.yaml"
# CONFIG = "src/data_challenge/configs/swin_s.yaml"
# ─────────────────────────────────────────────────────────────────────

with open(CONFIG) as f:
    cfg = yaml.safe_load(f)

# Point to Kaggle input directly — no copying needed
cfg["data"]["img_root"]  = img_root
cfg["data"]["train_csv"] = train_csv
cfg["data"]["test_csv"]  = test_csv
cfg["output"]["dir"]     = "/kaggle/working/outputs"

with open(CONFIG, "w") as f:
    yaml.dump(cfg, f)

print(f"Config   : {CONFIG}")
print(f"Backbone : {cfg['model']['backbone']}")
print(f"img_root : {cfg['data']['img_root']}")
print(f"Output   : {cfg['output']['dir']}")

In [ ]:
!PYTHONPATH=src python -m data_challenge.train --config {CONFIG}

## 5. Save Checkpoint to WandB Artifact

In [ ]:
import glob, json, wandb

summaries = glob.glob("/kaggle/working/outputs/*/run_summary.json")
if not summaries:
    print("No completed run found — training may have failed. Skipping artifact upload.")
else:
    summary_path = summaries[0]
    best_ckpt   = glob.glob("/kaggle/working/outputs/*/best_model.pt")[0]
    config_path = glob.glob("/kaggle/working/outputs/*/config.yaml")[0]

    with open(summary_path) as f:
        summary = json.load(f)

    run = wandb.init(
        project="data-challenge-occlusion",
        id=summary["wandb_run_id"],
        resume="must",
    )
    artifact = wandb.Artifact(summary["run_name"], type="model", metadata=summary)
    artifact.add_file(best_ckpt,    name="best_model.pt")
    artifact.add_file(summary_path, name="run_summary.json")
    artifact.add_file(config_path,  name="config.yaml")
    run.log_artifact(artifact)
    run.finish()

    print(f"Artifact '{summary['run_name']}' saved to wandb.")
    print(f"Best score: {summary['best_val_score']:.6f} at epoch {summary['best_epoch']}")

## 6. Pull checkpoint back to local machine

Run this on your **local machine** after the Kaggle run finishes:

```bash
.venv/bin/python -c "
import wandb
api = wandb.Api()
artifact = api.artifact('lehoang510/data-challenge-occlusion/dinov2_small:latest')
artifact.download('outputs/dinov2_small_kaggle/')
print('Done.')
"
```